# 06 - Oxygen case study: adapting the workflow to a new sensor

Case Study 2: dissolved oxygen (BTGOXD2 sensor, mg/L) at the same Tongoy
Balsa station, treated as a second, independent target for the same
pipeline used for chlorophyll in notebooks 01-05. This notebook is both a
**methodological adaptation guide** (Part A: what scientific decisions
have to be re-made for a new sensor, not just which columns to rename) and
the **worked result** of following it (Part B: the actual released oxygen
benchmark).

Part A's code cells that load data, audit coverage, generate an
illustrative artificial-gap sample, and score baselines are real and
executed: they run against `data/oxygen/`, the actual second sensor at
this site, using the same `src/coastal_gap_reconstruction/` functions as
the chlorophyll case study (called with `target_col="oxygen_mean_mgL"`,
`eligible_col="eligible_ge_18"` instead of the chlorophyll defaults). That
demonstrates the generic masking/scoring utilities run unchanged against a
second target -- it is not a claim that transferring a target is an
automated process, and the artificial-gap sample generated in Part A.3
below is illustrative only, not the released oxygen validation pool used
in Part B.

## Part A.0 Confirm this repository's scope

Read `docs/methods.md` and `docs/evidence_and_limitations.md` before
adapting anything. The target definition, eligibility rule, and validation
protocol are the load-bearing scientific decisions in this workflow;
copying the code without understanding these choices risks silently
producing invalid results for a new sensor.

## Part A.1 Define the new target

For a new sensor (e.g. oxygen), you will need to make the same decisions
documented in `docs/methods.md` for chlorophyll:

- What is the daily aggregation rule (mean? median?) and why?
- What hourly-validity threshold defines an "eligible" day?
- What is the full valid range, and how are invalid/negative/out-of-range
  raw readings handled?
- Are there known sensor drift, calibration, or fouling issues specific to
  this variable that need a different QA approach than chlorophyll?

These are scientific decisions, not implementation details -- they should
be made deliberately and documented, not inherited by default from the
chlorophyll target's design choices. For oxygen: raw mg/L throughout, **no
log10 transform** (unlike chlorophyll) -- oxygen legitimately approaches
zero under hypoxia, where log10 is unstable exactly at that physically
meaningful low tail.

In [1]:
# Live example: oxygen is the real second sensor this checklist produced. This cell
# actually loads it (no placeholder path) -- see data/oxygen/oxygen_daily_target.csv
# for the column names used below.

import sys

sys.path.insert(0, "../src")
from coastal_gap_reconstruction.data_loading import load_daily_target

OXYGEN_TARGET_COL = "oxygen_mean_mgL"
OXYGEN_ELIGIBLE_COL = "eligible_ge_18"

oxygen_df = load_daily_target("../data/oxygen/oxygen_daily_target.csv")
oxygen_df[[OXYGEN_TARGET_COL, OXYGEN_ELIGIBLE_COL]].head()

,oxygen_mean_mgL,eligible_ge_18
date,,
2015-07-01,NaN,False
2015-07-02,NaN,False
2015-07-03,NaN,False
2015-07-04,NaN,False
2015-07-05,NaN,False


## Part A.2 Re-run the gap audit

Use notebook 01 as a template: regenerate coverage statistics, real-gap
inventory, and eligible-run structure for the new target. Missingness
patterns may differ substantially between sensors (e.g. oxygen sensors may
have different fouling/drift failure modes than chlorophyll fluorometers).

In [2]:
from coastal_gap_reconstruction.gap_detection import coverage_summary

coverage_summary(oxygen_df, eligible_col=OXYGEN_ELIGIBLE_COL)

{'n_days_total': 3988,
 'n_days_eligible': 2880,
 'eligible_fraction': 0.7221664994984955,
 'n_real_gaps': 125,
 'longest_real_gap_days': 256}

## Part A.3 Sample an illustrative artificial-gap pool

Use `src/coastal_gap_reconstruction/artificial_gap_validation.py`'s
`generate_gap_candidates` function against the new target's eligible-run
structure. The gap lengths below (`[1, 3, 7, 14, 30]`) are a short
illustrative subset chosen for this notebook, not the released oxygen
benchmark's gap-length set. The actual released oxygen pool
(`data/oxygen/oxygen_validation_gaps.csv`) uses a primary support of
`[1, 3, 7, 10, 14, 21, 30]` days plus exploratory extended lengths
`[45, 60, 90, 120]`, sampled with additional rules not reproduced by this
generic function -- see `docs/methods.md`. The exact, byte-for-byte
reproduction of the released oxygen pool is
`experiments/oxygen/target_and_gap_pool.py`, not this illustrative cell.
Deciding the right gap-length set for a new sensor is itself one of the
scientific decisions this checklist flags, not something to copy from
chlorophyll or from this illustrative default.

In [3]:
from coastal_gap_reconstruction.artificial_gap_validation import generate_gap_candidates

oxygen_gap_pool = generate_gap_candidates(
    oxygen_df,
    gap_lengths=[1, 3, 7, 14, 30],
    target_col=OXYGEN_TARGET_COL,
    eligible_col=OXYGEN_ELIGIBLE_COL,
)
oxygen_gap_pool["gap_length"].value_counts().sort_index()

gap_length
1     100
3     100
7     100
14    100
30     48
Name: count, dtype: int64

## Part A.4 Re-evaluate baselines first

Use notebook 02's logic against the new target before trying anything more
sophisticated. The relative ranking of climatology vs. persistence vs.
interpolation may differ for a variable with different
seasonal/autocorrelation structure than chlorophyll.

In [4]:
import pandas as pd

from coastal_gap_reconstruction.artificial_gap_validation import apply_artificial_gap
from coastal_gap_reconstruction.baseline_imputation import run_all_baselines
from coastal_gap_reconstruction.scoring_metrics import compute_gap_metrics

sample_gap = oxygen_gap_pool.iloc[0]
masked = apply_artificial_gap(
    oxygen_df, sample_gap["start_date"], int(sample_gap["gap_length"]), target_col=OXYGEN_TARGET_COL
)
preds = run_all_baselines(
    masked, sample_gap["start_date"], int(sample_gap["gap_length"]),
    target_col=OXYGEN_TARGET_COL, eligible_col=OXYGEN_ELIGIBLE_COL,
)
rows = compute_gap_metrics(
    oxygen_df, preds, sample_gap["start_date"], int(sample_gap["gap_length"]),
    sample_gap["gap_id"], sample_gap.to_dict(), target_col=OXYGEN_TARGET_COL,
)
pd.DataFrame(rows)[["method", "mae", "rmse", "bias", "coverage"]]

,method,mae,rmse,bias,coverage
0,clim_monthly,0.7588,0.7588,-0.7588,1.0
1,persistence,1.1081,1.1081,1.1081,1.0
2,linear_interp,0.2849,0.2849,0.2849,1.0


## Part A.5 Re-select predictor features

The curated chlorophyll feature table includes chlorophyll-specific
covariates (a satellite chlorophyll proxy, upwelling indices tuned for
biological productivity). For a new sensor, re-evaluate which external
predictors are physically relevant -- do not assume the same feature table
transfers without justification. For oxygen: satellite chlorophyll is
exploratory/ablation-only (weak correlation, r=0.045-0.135), in-situ
chlorophyll is never admissible, and local water-temperature/pressure
readings from the *same buoy* as the oxygen sensor are restricted to one
clearly labeled diagnostic arm (co-missingness risk -- see Part B below).

## Part A.6 Re-run engineered tabular / TS-ICL methods

Once a baseline floor and a relevant feature set exist for the new sensor,
revisit `docs/methods.md` and notebook 04 to apply the same model ladder.
Re-check whether a satellite proxy covariate is meaningful for the new
variable -- for oxygen, for example, there is no obvious "satellite oxygen
proxy" analogous to satellite chlorophyll, so the leading TS-ICL
configuration for chlorophyll does not transfer directly (oxygen's leading
arm instead uses physical covariates -- SST, wind, solar, currents; see
Part B.4 below).

## Part A.7 Re-establish the evidence hierarchy

Apply the same discipline described in `docs/evidence_and_limitations.md`:
artificial-gap validation results are the only validation-grade evidence;
real-gap candidate outputs are plausibility only. This discipline does not
change across sensors. (Oxygen has a real-gap inventory but no
reconstruction-candidate generator -- Part B.7 below.)

---

## Part B.1 The released oxygen case study

Everything below loads the public daily dissolved-oxygen target table and
benchmark results for Tongoy Balsa (BTGOXD2 sensor, mg/L) -- the actual
result of following the Part A checklist above. Fully executable on the
public data included in this repository.

In [5]:
DATA_DIR = "../data/oxygen"
RESULTS_DIR = "../results/oxygen"

target_df = pd.read_csv(f"{DATA_DIR}/oxygen_daily_target.csv", parse_dates=["date"])
target_df = target_df.set_index("date").sort_index()
target_df.head()

,n_expected_hours,n_rows,valid_hours,coverage_fraction,quality_class,eligible_ge_12,eligible_ge_17,eligible_ge_18,eligible_ge_22,oxygen_mean_mgL,...,raw_prom_na_count,negative_oxygen_count,zero_oxygen_count,invalid_oxygen_count,first_valid_hour,last_valid_hour,longest_invalid_run_hours,source_variable,unit,timezone_status
date,,,,,,,,,,,,,,,,,,,,,
2015-07-01,24,24,0,0.0,missing,False,False,False,False,NaN,...,24,0,0,0,NaN,NaN,24,BTGOXD2,mg/L,reported_gmt_minus_4_unconverted
2015-07-02,24,24,0,0.0,missing,False,False,False,False,NaN,...,24,0,0,0,NaN,NaN,24,BTGOXD2,mg/L,reported_gmt_minus_4_unconverted
2015-07-03,24,24,0,0.0,missing,False,False,False,False,NaN,...,24,0,0,0,NaN,NaN,24,BTGOXD2,mg/L,reported_gmt_minus_4_unconverted
2015-07-04,24,24,0,0.0,missing,False,False,False,False,NaN,...,24,0,0,0,NaN,NaN,24,BTGOXD2,mg/L,reported_gmt_minus_4_unconverted
2015-07-05,24,24,0,0.0,missing,False,False,False,False,NaN,...,24,0,0,0,NaN,NaN,24,BTGOXD2,mg/L,reported_gmt_minus_4_unconverted


## Part B.2 Coverage summary

Eligibility mirrors the chlorophyll rule (>=18 valid hours/day required for
a trustworthy daily mean). Column `eligible_ge_18` is the eligibility flag
used throughout the oxygen benchmark.

In [6]:
n_days = len(target_df)
n_eligible = int(target_df["eligible_ge_18"].sum())
coverage_summary_ox = pd.Series({
    "n_days_total": n_days,
    "n_days_eligible": n_eligible,
    "pct_eligible": round(100 * n_eligible / n_days, 1),
    "n_days_missing_or_ineligible": n_days - n_eligible,
    "date_min": target_df.index.min().date(),
    "date_max": target_df.index.max().date(),
})
coverage_summary_ox

n_days_total                          3988
n_days_eligible                       2880
pct_eligible                          72.2
n_days_missing_or_ineligible          1108
date_min                        2015-07-01
date_max                        2026-05-31
dtype: object

## Part B.3 Real gap structure

Naturally occurring missing-data periods, grouped by length class. Real
gaps have no withheld ground truth -- see `docs/evidence_and_limitations.md`.

In [7]:
real_gaps = pd.read_csv(f"{DATA_DIR}/oxygen_real_gap_inventory_by_class.csv")
real_gaps

,gap_class,length_range_days,n_gaps,total_missing_days,median_length_days,max_length_days
0,short,1-7,105,146,1.0,6
1,medium,8-30,10,151,14.0,25
2,long,31-90,8,394,46.5,71
3,very_long,>90,2,417,208.5,256


## Part B.4 Artificial-gap validation pool

Withheld-truth validation gaps used to score every method. The **primary**
support is L=1-30 days (406 gaps) -- a more compact range than
chlorophyll's L=1-60, reflecting the shorter/denser missingness structure
of the oxygen record. The pool also carries 6 **exploratory extended**
gaps at L=45/60/90/120, not used in any benchmark number below (too few
gaps at each length for a meaningful comparison). See the `support_role`
column and `docs/data_dictionary.md`.

In [8]:
gap_pool = pd.read_csv(f"{DATA_DIR}/oxygen_validation_gaps.csv")
print(f"Total artificial gaps: {len(gap_pool)}")
if "support_role" in gap_pool.columns:
    print(gap_pool["support_role"].value_counts())
gap_pool["gap_length"].value_counts().sort_index() if "gap_length" in gap_pool.columns else gap_pool.head()

Total artificial gaps: 412
support_role
primary                 406
exploratory_extended      6
Name: count, dtype: int64


gap_length
1      100
3      100
7       78
10      55
14      36
21      22
30      15
45       3
60       1
90       1
120      1
Name: count, dtype: int64

## Part B.5 Benchmark summary

Mean absolute error (mg/L) by gap length for each method family, on the
oxygen artificial-gap validation pool. `TS-ICL physical covariates` (SST +
wind + solar + current) is the only method that clearly improves on
linear interpolation in the pooled benchmark -- 8.0% lower MAE, 95% CI
[4.5%, 11.4%].

In [9]:
bench = pd.read_csv(f"{RESULTS_DIR}/oxygen_benchmark_by_length.csv")
bench_pivot = bench.pivot_table(
    index="gap_length", columns="method_label", values="mae_gapweighted"
)
bench_pivot

method_label,External tabular (ExtraTrees),Gap-edge residual (ExtraTrees),Gaussian process,Linear interpolation,TS-ICL physical covariates,TS-ICL target-only
gap_length,,,,,,
1,0.605955,0.3802,0.345192,0.339383,0.300247,0.306787
3,0.805004,0.6740,0.642440,0.635099,0.561586,0.617770
7,0.949444,0.8760,0.860995,0.862560,0.824305,0.846076
10,0.962438,0.9705,0.969598,0.969838,0.930051,0.949336
14,0.997569,1.0935,1.086333,1.077817,0.949367,1.026675
21,1.102259,1.0054,0.963368,0.965973,0.944782,0.992032
30,1.040120,1.3061,1.345753,1.311827,1.153420,1.171420


In [10]:
deltas = pd.read_csv(f"{RESULTS_DIR}/oxygen_paired_deltas_vs_tsicl_physical_covariates.csv")
deltas[["comparator_label", "n_gaps", "delta_tsicl_minus_comparator_mae", "ci_lo", "ci_hi", "ci_excludes_zero"]]

,comparator_label,n_gaps,delta_tsicl_minus_comparator_mae,ci_lo,ci_hi,ci_excludes_zero
0,Linear interpolation,406,-0.058876,-0.086252,-0.032018,True
1,External tabular (ExtraTrees),406,-0.172303,-0.228687,-0.121047,True
2,Gaussian process,406,-0.063649,-0.091528,-0.036442,True
3,Gap-edge residual (ExtraTrees),406,-0.084483,-0.113936,-0.056405,True
4,TS-ICL target-only,406,-0.032325,-0.050624,-0.014354,True


## Part B.6 Local-BTG arm: diagnostic-only, never a core predictor result

`local_btg_temp_pressure_diagnostic` (water temperature + pressure,
BTGTA/BTGPA) is measured on the *same physical buoy* as the oxygen sensor
(BTGOXD2) itself. Its availability may covary with oxygen sensor outages
for reasons unrelated to any genuine physical relationship -- a day with
no oxygen reading is more likely to also have no BTGTA/BTGPA reading. For
this reason this arm is restricted to diagnostic/appendix use throughout
this package (`experiments.oxygen.benchmark_contract.LOCAL_BTG_TEMP_PRESSURE_ROLE`)
and must never be presented as a reliably available predictor during an
oxygen-sensor failure, even though it does show a real, CI-significant
TS-ICL improvement in isolation (+5.3%; see the report,
`manuscript/report/`, for the full same-station-diagnostic discussion).

## Part B.7 Distribution-tail limitation

TS-ICL's pooled improvement is not uniform across the oxygen distribution:
significant gains in the p10-p50 range, a significant loss in the high
tail (above p90, -18.8%), and a non-significant loss in the low tail
(below p10, -9.8%). See `results/oxygen/oxygen_tail_quantile_band_metrics.csv`,
`oxygen_tail_persistence_metrics.csv`, and
`manuscript/report/figures/fig_oxygen_tail_diagnostics_v6.pdf` (Section 5.2
of the report has the full discussion).

Oxygen also has a real-gap **inventory only** (Part B.3 above, 125 gaps) --
no reconstruction-candidate generator exists for oxygen real gaps; this is
a confirmed absence (`experiments/oxygen/real_gap_contract.py`), not an
oversight, unlike chlorophyll's two candidate methods in notebook 05.

## Part B.8 Reproducing and extending this case study

Every table above is the **frozen, released result** -- the authoritative
number for its method, not regenerated by this notebook. The full pipeline
that could regenerate them (`experiments/oxygen/`: `benchmark_contract.py`,
`feature_registry.py`, `classical_models.py`, `tsicl_models.py`,
`tail_diagnostics.py`, `run_oxygen_benchmark.py`) is published and tested,
at three reproducibility levels (see `docs/reproducibility.md`):

- **QUICK** (minutes, no live TS-ICL checkpoint needed): the test suite,
  this notebook, and frozen-result inspection --
  `python -m experiments.oxygen.run_oxygen_benchmark --mode frozen`.
- **BOUNDED** (well under an hour): the classical Model-0/GP comparators on
  the complete 406-gap primary support (`--mode classical`, ~30s, reproduces
  the frozen Model-0/GP numbers in Part B.5 above almost exactly) and a
  deterministic stratified live TS-ICL subset of at most 20 gaps
  (`--mode tsicl-bounded`, at most 60 real inference calls) that validates
  checkpoint loading, raw mg/L input/output, tensor shapes, and leakage
  safety -- **not** a new headline performance number, since the subset is
  small and length-biased.
- **EXPENSIVE, optional**: the complete TS-ICL grid (5 audited-original
  arms x 2 context modes + 4 exploratory family-ablation arms, all 406
  primary gaps) is not re-run by this package -- the frozen tables above
  are the authoritative complete result, exactly as with chlorophyll's own
  full covariate dissection.

Figures for this case study are sourced directly from the published report
(`manuscript/report/figures/`), which uses report-specific multi-panel
layouts, rather than regenerated by a standalone script.